In [ ]:
import dask
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display, HTML
from skimage.metrics import structural_similarity as ssim
from scipy.stats import pearsonr


In [ ]:
#!pip3 install scikit-image

In [ ]:
"""experiment_stats.py — compare reconstruction quality across experiments."""

from __future__ import annotations

import traceback
from typing import Any

import dask
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import HTML, display
from scipy.stats import pearsonr
from skimage.metrics import structural_similarity as ssim


# ── helpers ────────────────────────────────────────────────────────────────────

def _to_scalar(v: Any) -> Any:
    """numpy scalar / 0-d array → native Python type; leaves others alone."""
    if isinstance(v, np.ndarray):
        return v.flat[0].item()
    return v.item() if hasattr(v, "item") else v


def _to_scalar_list(arr) -> list:
    return [_to_scalar(v) for v in arr]


def compute_fid_score(orig_flat: np.ndarray, recon_flat: np.ndarray) -> float:
    mu1, sigma1 = orig_flat.mean(), orig_flat.std()
    mu2, sigma2 = recon_flat.mean(), recon_flat.std()
    return float((mu1 - mu2) ** 2 + (sigma1 - sigma2) ** 2)


# ── metric kernel ──────────────────────────────────────────────────────────────

def _metrics(orig_2d: np.ndarray, recon_2d: np.ndarray, value_range: float) -> dict:
    """Compute all reconstruction metrics for a single (variable, sample) slice."""
    diff     = orig_2d - recon_2d
    abs_diff = np.abs(diff)
    rmse     = float(np.sqrt(np.mean(diff * diff)))
    mae      = float(abs_diff.mean())
    max_ae   = float(abs_diff.max())
    bias     = float(diff.mean())
    diff_min = float(diff.min())
    diff_max = float(diff.max())
    ssim_v, _ = ssim(orig_2d, recon_2d, data_range=value_range, full=True)
    pcc, _    = pearsonr(orig_2d.ravel(), recon_2d.ravel())
    psnr      = float(20 * np.log10(value_range / rmse)) if rmse > 0 else np.inf
    afid      = compute_fid_score(orig_2d.ravel(), recon_2d.ravel())
    return dict(
        # ── error metrics ──────────────────────────────────────────────────
        RMSE=rmse, MAE=mae, MaxAE=max_ae, Bias=bias,
        DiffMin=diff_min, DiffMax=diff_max,
        # ── perceptual / correlation metrics ───────────────────────────────
        SSIM=float(ssim_v), PCC=float(pcc),
        PSNR=psnr, ApproxFID=afid,
        # ── original field statistics ──────────────────────────────────────
        Orig_Min=float(orig_2d.min()),
        Orig_Max=float(orig_2d.max()),
        Orig_Mean=float(orig_2d.mean()),
        Orig_Std=float(orig_2d.std()),
        # ── reconstruction field statistics ───────────────────────────────
        Recon_Min=float(recon_2d.min()),
        Recon_Max=float(recon_2d.max()),
        Recon_Mean=float(recon_2d.mean()),
        Recon_Std=float(recon_2d.std()),
    )


# ── styling ────────────────────────────────────────────────────────────────────

_LOWER_BETTER  = ["RMSE", "MAE", "MaxAE", "ApproxFID"]
_HIGHER_BETTER = ["SSIM", "PCC", "PSNR"]
_ZERO_BETTER   = ["DiffMin", "DiffMax", "Bias"]
# field-stat columns: no best/worst colouring (informational only)
_INFO_COLS     = ["Orig_Min", "Orig_Max", "Orig_Mean", "Orig_Std",
                  "Recon_Min", "Recon_Max", "Recon_Mean", "Recon_Std"]

_FMT = dict(
    RMSE="{:.5f}", MAE="{:.5f}", MaxAE="{:.5f}",
    Bias="{:+.5f}", DiffMin="{:+.5f}", DiffMax="{:+.5f}",
    SSIM="{:.4f}", PCC="{:.4f}", PSNR="{:.2f}", ApproxFID="{:.6f}",
    Orig_Min="{:.4f}",  Orig_Max="{:.4f}",  Orig_Mean="{:.4f}",  Orig_Std="{:.4f}",
    Recon_Min="{:.4f}", Recon_Max="{:.4f}", Recon_Mean="{:.4f}", Recon_Std="{:.4f}",
)

_TABLE_STYLES = [
    {"selector": "caption",
     "props": [("font-size","15px"),("font-weight","bold"),
               ("text-align","left"),("padding-bottom","6px")]},
    {"selector": "th",
     "props": [("background-color","#1e3a6e"),("color","white"),
               ("font-size","11px"),("text-align","center"),
               ("padding","5px 10px"),("white-space","nowrap")]},
    {"selector": "th.row_heading",
     "props": [("background-color","#3a5a9e"),("text-align","left"),
               ("max-width","280px"),("word-break","break-word")]},
    {"selector": "td",
     "props": [("font-size","11px"),("text-align","center"),
               ("padding","3px 10px")]},
    {"selector": "tr:hover td",
     "props": [("background-color","#eef2ff")]},
]

_LEGEND = """
<div style='font-size:11px;margin-top:2px;margin-bottom:22px;
            font-family:sans-serif;color:#555;line-height:1.8'>
  <b>Colour guide:</b>
  <span style='color:#1a7a1a;font-weight:bold'>■ best</span>&nbsp;
  <span style='color:#b30000;font-weight:bold'>■ worst</span>
  &nbsp;per column &nbsp;|&nbsp;
  <b>RMSE / MAE / MaxAE / FID</b>: lower = better &nbsp;|&nbsp;
  <b>SSIM / PCC / PSNR</b>: higher = better &nbsp;|&nbsp;
  <b>Bias / DiffMin / DiffMax</b>: |·| closer to 0 = better<br>
  <b>DiffMin</b>: most negative pixel in (orig−recon) — large negative = model over-predicts peaks &nbsp;|&nbsp;
  <b>DiffMax</b>: most positive pixel in (orig−recon) — large positive = model under-predicts peaks<br>
  <b>Orig_* / Recon_*</b>: per-grid statistics of the original and reconstruction fields (informational, no best/worst colouring)
</div>
"""


def _style_df(df: pd.DataFrame) -> "pd.io.formats.style.Styler":

    def _cell_styles(col: pd.Series, low: bool) -> pd.Series:
        styles = pd.Series("", index=col.index)
        v = col.dropna()
        if len(v) < 2:
            return styles
        best  = v.idxmin() if low else v.idxmax()
        worst = v.idxmax() if low else v.idxmin()
        styles[best]  = "font-weight:bold;color:#1a7a1a;background:#e6f4e6"
        styles[worst] = "font-weight:bold;color:#b30000;background:#fde8e8"
        return styles

    def _zero_styles(col: pd.Series) -> pd.Series:
        styles = pd.Series("", index=col.index)
        v = col.dropna()
        if len(v) < 2:
            return styles
        best  = v.abs().idxmin()
        worst = v.abs().idxmax()
        styles[best]  = "font-weight:bold;color:#1a7a1a;background:#e6f4e6"
        styles[worst] = "font-weight:bold;color:#b30000;background:#fde8e8"
        return styles

    def _info_styles(col: pd.Series) -> pd.Series:
        # Soft tint for informational field-stat columns — no best/worst.
        return pd.Series("background:#f5f0e8", index=col.index)

    styler = df.style
    for c in _LOWER_BETTER:
        if c in df.columns:
            styler = styler.apply(lambda s, c=c: _cell_styles(s, True),  subset=[c])
    for c in _HIGHER_BETTER:
        if c in df.columns:
            styler = styler.apply(lambda s, c=c: _cell_styles(s, False), subset=[c])
    for c in _ZERO_BETTER:
        if c in df.columns:
            styler = styler.apply(_zero_styles, subset=[c])
    for c in _INFO_COLS:
        if c in df.columns:
            styler = styler.apply(_info_styles, subset=[c])
    return styler


def _squeeze_size1_dims(da: xr.DataArray, keep: tuple[str, ...]) -> xr.DataArray:
    """Drop every size-1 dim not in `keep` via isel — works for uncoordinated dims."""
    for dim in list(da.dims):
        if dim in keep:
            continue
        if da.sizes[dim] == 1:
            da = da.isel({dim: 0}, drop=True)
    return da


# ── main function ──────────────────────────────────────────────────────────────

def print_experiment_stats(
    file_paths: dict,
    variables: list | None = None,
    samples: list | None = None,
    epoch_idx: int = -1,
) -> None:
    """
    Compare multiple experiments side-by-side in a styled Jupyter table.

    Dataset shape expected
    ----------------------
    (epoch, sample, time, variable, ensemble, y, x)
    where time=1 and ensemble=1 are size-1 dims (may lack coordinates).

    Parameters
    ----------
    file_paths : {experiment_label: zarr_path}
    variables  : variable name strings to include, or None for all
    samples    : sample indices to include, or None for all
    epoch_idx  : positional index into ds.epoch (-1 = last epoch)
    """

    # ── open stores lazily ────────────────────────────────────────────────────
    datasets: dict[str, xr.Dataset] = {}
    for label, path in file_paths.items():
        try:
            datasets[label] = xr.open_zarr(path)
        except Exception as exc:
            print(f"⚠️  Could not open '{label}': {exc}")

    if not datasets:
        print("No datasets loaded.")
        return

    first_ds = next(iter(datasets.values()))

    # Resolve variable names: use string coord if available, else caller list,
    # else positional labels "0", "1", …
    raw_var_vals = _to_scalar_list(first_ds.variable.values)
    if raw_var_vals and isinstance(raw_var_vals[0], str):
        all_vars = raw_var_vals
    else:
        n_vars   = int(first_ds.sizes["variable"])
        all_vars = variables if variables is not None else [str(i) for i in range(n_vars)]

    all_samps = _to_scalar_list(first_ds.sample.values)

    use_vars  = all_vars  if variables is None else [v for v in variables  if v in all_vars]
    use_samps = all_samps if samples   is None else [s for s in samples    if s in all_samps]

    # Fixed positional map: variable name → axis index in the variable dim
    var_pos_map: dict[str, int] = {v: i for i, v in enumerate(all_vars)}

    print(f"  Variables : {use_vars}")
    print(f"  Samples   : {use_samps}")

    # ── load one epoch per experiment ─────────────────────────────────────────
    print("Loading data (one batch per experiment)…")
    loaded: dict[str, dict] = {}

    for label, ds in datasets.items():
        try:
            epoch_pos = int(epoch_idx) % len(ds.epoch)
            epoch_val = _to_scalar(ds.epoch.values[epoch_pos])

            samp_coord     = ds.sample.values
            samp_positions: list[int] = []
            valid_samps:    list      = []
            for s in use_samps:
                hits = np.where(samp_coord == s)[0]
                if len(hits):
                    samp_positions.append(int(hits[0]))
                    valid_samps.append(s)

            if not samp_positions:
                raise ValueError("None of the requested samples found in dataset.")

            use_var_positions = [var_pos_map[v] for v in use_vars if v in var_pos_map]

            def _extract(data_var: str) -> xr.DataArray:
                da = ds[data_var].isel(
                    epoch=epoch_pos,
                    sample=samp_positions,
                    variable=use_var_positions,
                )
                da = _squeeze_size1_dims(da, keep=("sample", "variable", "y", "x"))
                return da.transpose("sample", "variable", "y", "x")

            orig_da, recon_da = _extract("original"), _extract("reconstruction")

            # Single dask graph pass for both arrays
            orig_np, recon_np = dask.compute(orig_da, recon_da)
            orig_arr  = orig_np.values.astype(np.float32)   # (S, V, Y, X)
            recon_arr = recon_np.values.astype(np.float32)

            loaded[label] = {
                "epoch":      epoch_val,
                "orig":       orig_arr,
                "recon":      recon_arr,
                "var_index":  {v: i for i, v in enumerate(use_vars) if v in var_pos_map},
                "samp_index": {s: i for i, s in enumerate(valid_samps)},
            }
            print(f"  ✓ {label}  (epoch {epoch_val})")

        except Exception as exc:
            print(f"  ⚠️  {label}: {exc}")
            traceback.print_exc()

    if not loaded:
        print("No experiments loaded successfully.")
        return

    # ── value_range per variable (across all experiments) ────────────────────
    value_ranges: dict[str, float] = {}
    for var in use_vars:
        v_min, v_max = np.inf, -np.inf
        for info in loaded.values():
            vi = info["var_index"].get(var)
            if vi is None:
                continue
            v_min = min(v_min, float(info["orig"][:, vi].min()),
                                float(info["recon"][:, vi].min()))
            v_max = max(v_max, float(info["orig"][:, vi].max()),
                                float(info["recon"][:, vi].max()))
        value_ranges[var] = (v_max - v_min) if v_max > v_min else 1.0

    # ── compute metrics: experiment → variable → sample ───────────────────────
    print("Computing metrics…")
    # rows[var][sample][col_label] = metrics_dict
    rows: dict[str, dict] = {var: {s: {} for s in use_samps} for var in use_vars}

    for label, info in loaded.items():
        col_label = f"{label}\n(ep {info['epoch']})"
        for var in use_vars:
            vi = info["var_index"].get(var)
            if vi is None:
                continue
            vr = value_ranges[var]
            for sample in use_samps:
                si = info["samp_index"].get(sample)
                if si is None:
                    continue
                orig_2d  = info["orig"] [si, vi].astype(np.float64)
                recon_2d = info["recon"][si, vi].astype(np.float64)
                rows[var][sample][col_label] = _metrics(orig_2d, recon_2d, vr)

    # ── display tables per variable ───────────────────────────────────────────
    metric_keys: list[str] = list(_FMT.keys())
    for var in use_vars:
        for s in use_samps:
            if rows[var][s]:
                metric_keys = list(next(iter(rows[var][s].values())).keys())
                break

    for var in use_vars:
        rows_v  = rows[var]
        all_dfs = []

        for sample in use_samps:
            if not rows_v[sample]:
                continue
            df = pd.DataFrame(rows_v[sample]).T
            df.index.name = "Experiment (epoch)"
            all_dfs.append((f"Sample {sample} — var {var}", df))

        # Average table across samples
        if len(use_samps) > 1:
            all_col_labels = {col for s in use_samps for col in rows_v[s]}
            avg_rows: dict[str, dict] = {}
            for col_label in all_col_labels:
                vals = [rows_v[s][col_label] for s in use_samps if col_label in rows_v[s]]
                if vals:
                    avg_rows[col_label] = {
                        k: float(np.mean([v[k] for v in vals])) for k in metric_keys
                    }
            if avg_rows:
                df_avg = pd.DataFrame(avg_rows).T
                df_avg.index.name = "Experiment (epoch)"
                all_dfs.append((f"Avg across samples — var {var}", df_avg))

        if not all_dfs:
            continue

        display(HTML(
            f"<h3 style='margin-bottom:4px;font-family:sans-serif;"
            f"color:#1e3a6e;border-bottom:2px solid #1e3a6e;padding-bottom:4px'>"
            f"Variable: <code>{var}</code></h3>"
        ))
        for subtitle, df in all_dfs:
            styler = (_style_df(df)
                      .format(_FMT)
                      .set_caption(subtitle)
                      .set_table_styles(_TABLE_STYLES))
            display(styler)
        display(HTML(_LEGEND))

    print("Done.")

In [ ]:
import xarray as xr


file_paths = {
   #'02-l2-no-norm/mse_kl_04_0_20260609_024857':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/02-l2-no-norm/mse_kl_04_0_20260609_024857/samples.zarr',
   #'03-lola-no-norm/lola_dcae_20260609_024929':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/03-lola-no-norm/lola_dcae_20260609_024929/samples.zarr',
   #'04-qrl-no-norm/qrl_dcae_20260609_024937':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/04-qrl-no-norm/qrl_dcae_20260609_024937/samples.zarr',
   #'05-lola-uniform/lola_uniform_20260609_024945':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/05-lola-uniform/lola_uniform_20260609_024945/samples.zarr',
   #'06-lola-winds/lola_winds_20260609_024952':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/06-lola-winds/lola_winds_20260609_024952/samples.zarr',
   #'07-lola-32x/lola_32x_20260609_025000':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/07-lola-32x/lola_32x_20260609_025000/samples.zarr',
   '08-lola-z64/lola_z64_20260609_025027':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/08-lola-z64/lola_z64_20260609_025027/samples.zarr',
   #'09-winds-focal/winds_focal_20260609_025034':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/winds_focal_20260609_025034/samples.zarr',
   '09-winds-focal-z32x32x32-20260609_145310':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/09-winds-focal-z32x32x32-20260609_145310/samples.zarr',
   '09-winds-focal-z64x16x16-20260609_145317':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/09-winds-focal-z64x16x16-20260609_145317/samples.zarr',
   #'10-arcsinh-z64/arcsinh_z64_20260609_025040':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/10-arcsinh-z64/arcsinh_z64_20260609_025040/samples.zarr',
   #'11-qrl-32x/qrl_32x_20260609_025046':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/11-qrl-32x/qrl_32x_20260609_025046/samples.zarr',
   #'12-rombach-ae/kl_f8_20260609_032527':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/12-rombach-ae/kl_f8_20260609_032527/samples.zarr',
   #'12-rombach-ae/vq_f8_20260609_032535':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/12-rombach-ae/vq_f8_20260609_032535/samples.zarr',
   '13-qrl-vae-winds-focal-z64x16x16-20260609_150346':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/13-qrl-vae-winds-focal/13-qrl-vae-winds-focal-z64x16x16-20260609_150346/samples.zarr',
   #'15-qrl-vae-winds-kl-z32x32x32-20260610_102718':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/15-qrl-vae-winds-kl/15-qrl-vae-winds-kl-z32x32x32-20260610_102718/samples.zarr',
   #'15-qrl-vae-winds-kl-z64x16x16-20260610_102719':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/15-qrl-vae-winds-kl/15-qrl-vae-winds-kl-z64x16x16-20260610_102719/samples.zarr',
   #'16-qrl-vae-winds-kl-mmd-z32x32x32-20260610_102720':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/16-qrl-vae-winds-kl-mmd/16-qrl-vae-winds-kl-mmd-z32x32x32-20260610_102720/samples.zarr',
   #'16-qrl-vae-winds-kl-mmd-z64x16x16-20260610_102721':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/16-qrl-vae-winds-kl-mmd/16-qrl-vae-winds-kl-mmd-z64x16x16-20260610_102721/samples.zarr',
   #'17-lola-vae-kl-mmd-z64-20260610_102722/':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/17-lola-vae-kl/17-lola-vae-kl-mmd-z64-20260610_102722/samples.zarr',
   #'17-lola-vae-kl-z64-20260610_102724':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/17-lola-vae-kl/17-lola-vae-kl-z64-20260610_102724/samples.zarr',
}


In [ ]:
file_paths = {
   'mse_kl_04_0_20260417_092115':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae-answer/mse_kl_04_0_20260417_092115/samples.zarr',
   #'02-l2-no-norm/mse_kl_04_0_20260609_024857':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/02-l2-no-norm/mse_kl_04_0_20260609_024857/samples.zarr',
   #'03-lola-no-norm/lola_dcae_20260609_024929':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/03-lola-no-norm/lola_dcae_20260609_024929/samples.zarr',
   #'04-qrl-no-norm/qrl_dcae_20260609_024937':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/04-qrl-no-norm/qrl_dcae_20260609_024937/samples.zarr',
   #'05-lola-uniform/lola_uniform_20260609_024945':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/05-lola-uniform/lola_uniform_20260609_024945/samples.zarr',
   #'06-lola-winds/lola_winds_20260609_024952':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/06-lola-winds/lola_winds_20260609_024952/samples.zarr',
   #'07-lola-32x/lola_32x_20260609_025000':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/07-lola-32x/lola_32x_20260609_025000/samples.zarr',
      #'08-lola-z64/lola_z64_20260609_025027':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/08-lola-z64/lola_z64_20260609_025027/samples.zarr',
   #'09-winds-focal/winds_focal_20260609_025034':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/winds_focal_20260609_025034/samples.zarr',
      #'09-winds-focal-z32x32x32-20260609_145310':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/09-winds-focal-z32x32x32-20260609_145310/samples.zarr',
      #'09-winds-focal-z64x16x16-20260609_145317':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/09-winds-focal-z64x16x16-20260609_145317/samples.zarr',
   #'10-arcsinh-z64/arcsinh_z64_20260609_025040':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/10-arcsinh-z64/arcsinh_z64_20260609_025040/samples.zarr',
   #'11-qrl-32x/qrl_32x_20260609_025046':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/11-qrl-32x/qrl_32x_20260609_025046/samples.zarr',
   #'12-rombach-ae/kl_f8_20260609_032527':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/12-rombach-ae/kl_f8_20260609_032527/samples.zarr',
   #'12-rombach-ae/vq_f8_20260609_032535':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/12-rombach-ae/vq_f8_20260609_032535/samples.zarr',
   '13-qrl-vae-winds-focal-z64x16x16-20260609_150346':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/13-qrl-vae-winds-focal/13-qrl-vae-winds-focal-z64x16x16-20260609_150346/samples.zarr',
   #'15-qrl-vae-winds-kl-z32x32x32-20260610_102718':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/15-qrl-vae-winds-kl/15-qrl-vae-winds-kl-z32x32x32-20260610_102718/samples.zarr',
   #'15-qrl-vae-winds-kl-z64x16x16-20260610_102719':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/15-qrl-vae-winds-kl/15-qrl-vae-winds-kl-z64x16x16-20260610_102719/samples.zarr',
   #'16-qrl-vae-winds-kl-mmd-z32x32x32-20260610_102720':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/16-qrl-vae-winds-kl-mmd/16-qrl-vae-winds-kl-mmd-z32x32x32-20260610_102720/samples.zarr',
   #'16-qrl-vae-winds-kl-mmd-z64x16x16-20260610_102721':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/16-qrl-vae-winds-kl-mmd/16-qrl-vae-winds-kl-mmd-z64x16x16-20260610_102721/samples.zarr',
   #'17-lola-vae-kl-mmd-z64-20260610_102722/':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/17-lola-vae-kl/17-lola-vae-kl-mmd-z64-20260610_102722/samples.zarr',
   #'17-lola-vae-kl-z64-20260610_102724':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/17-lola-vae-kl/17-lola-vae-kl-z64-20260610_102724/samples.zarr',
}


In [ ]:
# ── run ───────────────────────────────────────────────────────────────────────
print_experiment_stats(
    file_paths,
    variables=None,   # None = all variables in the dataset
    samples=None,     # None = all samples
    epoch_idx=-1,     # -1   = last saved epoch
)